# Lesson 10 - AI Agents sa Produksyon

Sa leksyong ito malalaman mo ang **mga pattern ng produksyon** para sa AI agents gamit ang Microsoft Agent Framework (Python). Tatalakayin natin ang:

- **Observability** — pagdaragdag ng timing at logging sa mga pakikipag-ugnayan ng agent
- **Evaluation** — paggamit ng evaluator agent para i-score ang kalidad ng tugon
- **Cost management** — mga estratehiya para sa pag-optimize ng token at pagpili ng modelo

Ang senaryo ay isang **travel agent** na tumutulong sa mga user magplano ng mga biyahe, na may monitoring at evaluation na naka-layer sa itaas.


## Setup


In [ ]:
%pip install agent-framework azure-ai-projects azure-identity python-dotenv -U -q

In [ ]:
import logging
logging.getLogger("agent_framework.foundry").setLevel(logging.ERROR)

import os
import asyncio
import time
import dotenv
from typing import Annotated

from agent_framework import tool
from agent_framework.foundry import FoundryChatClient
from azure.identity import DefaultAzureCredential

dotenv.load_dotenv()

endpoint = os.getenv("AZURE_AI_PROJECT_ENDPOINT")
deployment_name = os.getenv("AZURE_AI_MODEL_DEPLOYMENT_NAME")

missing = [k for k, v in {
    "AZURE_AI_PROJECT_ENDPOINT": endpoint,
    "AZURE_AI_MODEL_DEPLOYMENT_NAME": deployment_name
}.items() if not v]

if missing:
    raise ValueError(
        f"Missing required environment variables: {', '.join(missing)}. "
        "Please set them as environment variables (e.g., in your .env file or shell environment)."
    )

In [ ]:
# Create the Azure AI Foundry client
client = FoundryChatClient(
    project_endpoint=endpoint,
    model=deployment_name,
    credential=DefaultAzureCredential()
)

## Mga Pagsasaalang-alang sa Produksyon

Ang paglipat ng mga AI agent mula sa mga prototype patungo sa produksyon ay nangangailangan ng maingat na pagtutok sa tatlong haligi:

1. **Observability** — Kailangan mo ng kakayahang makita kung ano ang ginagawa ng agent, gaano katagal ito, at alin sa mga tool ang tinatawag nito. Kung walang tracing at logging, halos imposible ang pag-debug ng mga isyu sa produksyon.

2. **Evaluation** — Ang mga automated na pagsusuri sa kalidad ay nagsisiguro na ang mga tugon ng agent ay nananatiling tama, kumpleto, at kapaki-pakinabang sa paglipas ng panahon. Ang isang evaluator agent ay maaaring magbigay ng iskor sa mga tugon batay sa mga itinakdang pamantayan.

3. **Cost Management** — Ang paggamit ng token ay direktang nakakaapekto sa gastos. Ang mga estratehiya tulad ng prompt optimization, pagpili ng modelo, at caching ay tumutulong upang mapanatili ang mga gastusin sa loob ng kontrol nang hindi isinusuko ang kalidad.


## Paggawa ng Isang Observable Agent

Tinukoy namin ang mga travel tools at binalot ang pagtawag sa agent gamit ang timing upang ma-monitor namin ang latency. Sa produksyon, iintegrate mo ito sa OpenTelemetry o katulad na tracing backend.


In [ ]:
@tool(approval_mode="never_require")
def get_flight_info(destination: Annotated[str, "The destination city"]) -> str:
    """Get flight information for a destination."""
    flights = {
        "Paris": "BA 304, 08:30-11:45, $350",
        "Tokyo": "JL 044, 11:00-07:00+1, $890",
        "Barcelona": "VY 7821, 07:15-10:30, $280",
    }
    return flights.get(destination, f"No flights found to {destination}")


@tool(approval_mode="never_require")
def get_activity_suggestions(destination: Annotated[str, "The destination city"]) -> str:
    """Get activity suggestions for a destination."""
    activities = {
        "Paris": "Louvre Museum, Eiffel Tower, Seine River Cruise, Montmartre walking tour",
        "Tokyo": "Senso-ji Temple, Tsukiji Market tour, Shibuya Crossing, teamLab Borderless",
        "Barcelona": "Sagrada Familia, Park Güell, La Boqueria Market, Gothic Quarter walk",
    }
    return activities.get(destination, f"No activities found for {destination}")

In [ ]:
agent = client.as_agent(
    tools=[get_flight_info, get_activity_suggestions],
    name="TravelAgent",
    instructions="You are a helpful travel agent. Use the available tools to help users plan their trips. Provide comprehensive, actionable travel advice.",
)

# Simple observability: track timing
start_time = time.time()
response = await agent.run(
    "I want to plan a day trip in Paris. What flights and activities do you recommend?",
    )
elapsed = time.time() - start_time
print(f"Response ({elapsed:.2f}s):\n{response}")

## Mga Pattern ng Pagsusuri

Isang karaniwang pattern sa produksyon ay ang paggamit ng pangalawang ahente bilang **tagapagpasiya**. Sinusukat ng tagapagpasiya ang tugon ng pangunahing ahente laban sa mga paunang natukoy na pamantayan tulad ng pagiging kumpleto, katumpakan, at kapaki-pakinabang.

Ito ay nagpapahintulot sa:
- Otomatikong mga gate ng kalidad bago makarating sa mga gumagamit ang mga tugon
- Pagtuklas ng regresyon kapag nagbago ang mga prompt o modelo
- Patuloy na pagmamanman ng pagganap ng ahente sa paglipas ng panahon


In [ ]:
evaluator = client.as_agent(
    name="ResponseEvaluator",
    instructions="""You evaluate travel agent responses on these criteria:
1. Completeness (1-5): Did it cover flights AND activities?
2. Accuracy (1-5): Is the information consistent?
3. Helpfulness (1-5): Would a traveler find this actionable?
4. Overall Score (1-5)
Provide scores and a brief explanation for each.""",
)

evaluation = await evaluator.run(f"Evaluate this travel agent response:\n\n{response}")
print(f"Evaluation:\n{evaluation}")

## Mga Estratehiya sa Pamamahala ng Gastos

Ang pagkontrol sa gastos ay mahalaga para sa mga production AI agent. Narito ang mga pangunahing estratehiya:

| Estratehiya | Paglalarawan |
|---|---|
| **Pag-optimize ng prompt** | Panatilihing maigsi ang mga instruksyon ng sistema. Alisin ang mga paulit-ulit na konteksto upang mabawasan ang mga input token. |
| **Pagpili ng modelo** | Gamitin ang mas maliliit at mas murang modelo (hal. GPT-4o-mini) para sa mga simpleng gawain tulad ng klasipikasyon o pagkuha, at gamitin ang mas malalaki para sa mga komplikadong pag-aanalisa. |
| **Caching** | I-cache ang mga resulta ng tool at madalas na mga query upang maiwasan ang paulit-ulit na tawag sa API. |
| **Token budgets** | Magtakda ng mga limitasyon sa `max_tokens` upang maiwasan ang hindi inaasahang mahabang sagot. |
| **Batching** | Pagsamahin ang maramihang mga query ng gumagamit sa isang tawag sa API kung saan posible. |

Sa praktis, epektibo ang sunud-sunod na pamamaraan: ituro ang mga tuwirang kahilingan sa mabilis at murang modelo at itaas lamang sa mas capable na modelo ang mga komplikadong query.


## Summary

Sa leksyong ito natutunan mo kung paano:

1. **Magdagdag ng observability** sa interaksyon ng ahente gamit ang timing at pag-log, na naglalatag ng pundasyon para sa tracing at monitoring.
2. **Suriin ang mga tugon ng ahente** nang awtomatiko gamit ang isang evaluator agent na nagbibigay ng puntos sa pagiging kumpleto, katumpakan, at kapaki-pakinabang.
3. **Pamahalaan ang mga gastos** sa pamamagitan ng pag-optimize ng prompt, pagpili ng modelo, pag-cache, at mga budget ng token.

Ang mga pattern na ito para sa produksyon ay tumutulong upang matiyak na ang iyong mga AI agent ay maaasahan, masusukat, at epektibo sa gastos sa malawakang aplikasyon.


---

<!-- CO-OP TRANSLATOR DISCLAIMER START -->
**Pagtatanggi**:
Ang dokumentong ito ay isinalin gamit ang serbisyo ng AI translation na [Co-op Translator](https://github.com/Azure/co-op-translator). Bagama't nagsusumikap kami para sa katumpakan, pakatandaan na ang awtomatikong pagsasalin ay maaaring maglaman ng mga pagkakamali o hindi pagkakatugma. Ang orihinal na dokumento sa orihinal nitong wika ang dapat ituring na pangunahing sanggunian. Para sa mahahalagang impormasyon, inirerekomenda ang propesyonal na pagsasalin ng tao. Hindi kami mananagot sa anumang maling pagkakaintindi o maling interpretasyon na nagmula sa paggamit ng pagsasaling ito.
<!-- CO-OP TRANSLATOR DISCLAIMER END -->
